# CineMind
## Semantic Movie Recommendation System

# 1. Project Introduction

CineMind is a content-based movie recommendation system. Given a movie a user likes, it recommends other movies that are similar in content, not based on what other users watched.

Content-based recommendation was chosen because the dataset does not contain user ratings or watch history, only movie information (overview, genre, popularity, etc.). This means the system has to find similarity from the movie's own content instead of user behavior.

The notebook documents the full journey of the project:
- a TF-IDF baseline
- an improved TF-IDF version with weighted genre
- a hybrid ranking approach (similarity + rating + popularity)
- the final model: Sentence Transformer embeddings + FAISS semantic search

The final model (Sentence Transformer + FAISS) is used because it understands the *meaning* of a movie's description, not just the exact words used, which gives noticeably better recommendations.

# 2. Import Required Libraries

In [ ]:
# pandas / numpy
import pandas as pd
import numpy as np

# text cleaning
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# TF-IDF and similarity
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# semantic search
from sentence_transformers import SentenceTransformer
import faiss

# file/path utilities
import os

# 3. Load the Dataset

### Loading the Dataset

The dataset used is `mymoviedb.csv`, a Kaggle dataset with movie title, overview, popularity, ratings, language, genre and poster URL for close to 10,000 movies.

In [ ]:
df = pd.read_csv(
    "mymoviedb.csv",
    on_bad_lines="skip",
    engine="python"
)

### Shape of the Dataset

Check the number of rows and columns.

In [ ]:
df.shape

(9837, 9)

### First Few Records

View the first few rows to understand the structure of the data.

In [ ]:
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


### Dataset Information

Check column names, data types, and non-null counts.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Release_Date       9837 non-null   object 
 1   Title              9828 non-null   object 
 2   Overview           9828 non-null   object 
 3   Popularity         9827 non-null   float64
 4   Vote_Count         9827 non-null   object 
 5   Vote_Average       9827 non-null   object 
 6   Original_Language  9827 non-null   object 
 7   Genre              9826 non-null   object 
 8   Poster_Url         9826 non-null   object 
dtypes: float64(1), object(8)
memory usage: 691.8+ KB


### Column Names

In [ ]:
df.columns

Index(['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count',
       'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url'],
      dtype='object')

### Missing Values

Check how many missing values each column has before doing any cleaning.

In [ ]:
df.isna().sum()

Release_Date          0
Title                 9
Overview              9
Popularity           10
Vote_Count           10
Vote_Average         10
Original_Language    10
Genre                11
Poster_Url           11
dtype: int64

# 4. Data Understanding

### Duplicate Movie Titles

Some movies share the same title (remakes, sequels named the same, re-releases). This is just for understanding the data — these rows are not removed, since they are different movies with the same name.

In [ ]:
duplicate_titles = df[df["Title"].duplicated(keep=False)].sort_values("Title")
print("Rows with duplicate titles:", len(duplicate_titles))
display(
    duplicate_titles[
        ["Title", "Release_Date", "Genre", "Vote_Average", "Poster_Url"]
    ].head(30)
)

Rows with duplicate titles: 602


,Title,Release_Date,Genre,Vote_Average,Poster_Url
3280,365 Days,2015-05-22,Romance,4.8,https://image.tmdb.org/t/p/original/AoFyZGDgfr...
1434,365 Days,2020-02-07,"Drama, Romance",7.0,https://image.tmdb.org/t/p/original/6KwrHucIE3...
6003,A Christmas Carol,1984-10-09,"Drama, Fantasy, TV Movie, Family",7.3,https://image.tmdb.org/t/p/original/kGmpIkjVRW...
2062,A Christmas Carol,2009-11-04,"Animation, Family, Drama, Fantasy",6.9,https://image.tmdb.org/t/p/original/goHDZUnqZJ...
2179,A Nightmare on Elm Street,2010-04-30,"Horror, Mystery, Thriller",5.5,https://image.tmdb.org/t/p/original/4W2GvVM89X...
1609,A Nightmare on Elm Street,1984-11-09,Horror,7.3,https://image.tmdb.org/t/p/original/wGTpGGRMZm...
3713,Adventures in Babysitting,2016-06-24,"TV Movie, Comedy, Family",7.4,https://image.tmdb.org/t/p/original/xuCPo0krpA...
5912,Adventures in Babysitting,1987-07-01,Comedy,6.8,https://image.tmdb.org/t/p/original/8QiAj0Xw2U...
1093,Aftermath,2021-08-04,"Horror, Crime, Drama, Thriller",6.5,https://image.tmdb.org/t/p/original/cbhxqeDIge...
3576,Aftermath,2017-02-28,"Drama, Thriller",5.7,https://image.tmdb.org/t/p/original/25oWTCGqIi...


# 5. Data Cleaning

### Removing Rows with Missing Titles

A movie with no title is not usable for recommendations, so these rows are dropped.

In [ ]:
df = df.dropna(subset=["Title"]).copy()
print("Shape after removing missing titles:", df.shape)

Shape after removing missing titles: (9828, 9)


### Checking Missing Values After Removing Missing Titles

In [ ]:
df.isna().sum()

Release_Date         0
Title                0
Overview             0
Popularity           1
Vote_Count           1
Vote_Average         1
Original_Language    1
Genre                2
Poster_Url           2
dtype: int64

### Filling Missing Text Values

`Genre`, `Original_Language`, `Overview` and `Poster_Url` are filled with an empty string so they don't break the text-cleaning steps later.

In [ ]:
df["Genre"] = df["Genre"].fillna("")
df["Original_Language"] = df["Original_Language"].fillna("")
df["Overview"] = df["Overview"].fillna("")
df["Poster_Url"] = df["Poster_Url"].fillna("")

### Filling Missing Numeric Values

`Popularity`, `Vote_Count` and `Vote_Average` are filled with 0.

In [ ]:
df["Popularity"] = df["Popularity"].fillna(0)
df["Vote_Count"] = df["Vote_Count"].fillna(0)
df["Vote_Average"] = df["Vote_Average"].fillna(0)

### Checking Missing Values After Cleaning

In [ ]:
df.isna().sum()

Release_Date         0
Title                0
Overview             0
Popularity           0
Vote_Count           0
Vote_Average         0
Original_Language    0
Genre                0
Poster_Url           0
dtype: int64

### Converting Numeric Columns

`Popularity`, `Vote_Count` and `Vote_Average` were loaded as text in some rows because of bad lines in the CSV, so they are converted to numeric.

In [ ]:
df["Popularity"] = pd.to_numeric(df["Popularity"], errors="coerce")
df["Vote_Count"] = pd.to_numeric(df["Vote_Count"], errors="coerce")
df["Vote_Average"] = pd.to_numeric(df["Vote_Average"], errors="coerce")
print(df[["Popularity", "Vote_Count", "Vote_Average"]].dtypes)

Popularity      float64
Vote_Count      float64
Vote_Average    float64
dtype: object


### Creating Unique Movie IDs

A simple integer `movie_id` is created from the row index. This id is used everywhere else in the notebook to look up a movie.

In [ ]:
df = df.reset_index(drop=True)
df["movie_id"] = df.index
print("Total Movies:", len(df))
print(df[["movie_id", "Title", "Release_Date"]].head(10))

Total Movies: 9828
   movie_id                    Title Release_Date
0         0  Spider-Man: No Way Home   2021-12-15
1         1               The Batman   2022-03-01
2         2                  No Exit   2022-02-25
3         3                  Encanto   2021-11-24
4         4           The King's Man   2021-12-22
5         5             The Commando   2022-01-07
6         6                   Scream   2022-01-12
7         7                     Kimi   2022-02-10
8         8     Fistful of Vengeance   2022-02-17
9         9                 Eternals   2021-11-03


### Verifying Unique IDs

In [ ]:
print("Unique Movie IDs:", df["movie_id"].nunique())
print("Total Movies:", len(df))

Unique Movie IDs: 9828
Total Movies: 9828


### Checking for Duplicate Movies

Check whether any movie appears more than once with the same title and release date.

In [ ]:
duplicate_movies = df.duplicated(
    subset=["Title", "Release_Date"],
    keep=False
)
print("Duplicate Title + Release Date:", duplicate_movies.sum())

Duplicate Title + Release Date: 0


# 6. Feature Engineering

### Preparing Text for Feature Extraction

Stopwords and a lemmatizer are needed to clean the `Overview` and `Genre` text before building any text-based features.

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arjun\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arjun\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Text Cleaning Function

Lowercases the text, removes punctuation and numbers, removes stopwords, and lemmatizes each word.

In [ ]:
def preprocess(text):

    text = str(text).lower()

    # remove punctuation & numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # tokenize
    words = text.split()

    # stopword removal + lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

### Applying Text Cleaning

Clean `Overview` and `Genre` into `Overview_clean` and `Genre_clean`.

In [ ]:
df["Overview_clean"] = df["Overview"].apply(preprocess)
df["Genre_clean"] = df["Genre"].apply(preprocess)

### Combining Movie Features into Tags

`tags` combines the cleaned overview, cleaned genre and original language into a single text field that represents each movie.

In [ ]:
df["tags"] = (
    df["Overview_clean"] + " " +
    df["Genre_clean"] + " " +
    df["Original_Language"]
)

### Preview of Tags

In [ ]:
df[[
    "Title",
    "Overview_clean",
    "Genre_clean",
    "tags"
]].head()

,Title,Overview_clean,Genre_clean,tags
0,Spider-Man: No Way Home,peter parker unmasked longer able separate nor...,action adventure science fiction,peter parker unmasked longer able separate nor...
1,The Batman,second year fighting crime batman uncovers cor...,crime mystery thriller,second year fighting crime batman uncovers cor...
2,No Exit,stranded rest stop mountain blizzard recoverin...,thriller,stranded rest stop mountain blizzard recoverin...
3,Encanto,tale extraordinary family madrigal live hidden...,animation comedy family fantasy,tale extraordinary family madrigal live hidden...
4,The King's Man,collection history worst tyrant criminal maste...,action adventure thriller war,collection history worst tyrant criminal maste...


### Creating Weighted Movie Features

`tags_v2` repeats `Genre_clean` three times before adding the overview. Genre is repeated so it carries more weight than the overview text when similarity is calculated — otherwise the much longer overview text would dominate.

In [ ]:
# Create weighted movie tags

df["tags_v2"] = (
    df["Genre_clean"] + " " +
    df["Genre_clean"] + " " +
    df["Genre_clean"] + " " +
    df["Overview_clean"] + " " +
    df["Original_Language"]
)

df[["Title", "tags_v2"]].head()

,Title,tags_v2
0,Spider-Man: No Way Home,action adventure science fiction action advent...
1,The Batman,crime mystery thriller crime mystery thriller ...
2,No Exit,thriller thriller thriller stranded rest stop ...
3,Encanto,animation comedy family fantasy animation come...
4,The King's Man,action adventure thriller war action adventure...


# 7. TF-IDF Baseline

### TF-IDF Vectorization

TF-IDF is used as the initial baseline. It represents each movie's `tags` as a vector of word/word-pair importance scores. Unigrams and bigrams are used (`ngram_range=(1, 2)`), and very rare or very common terms are filtered out with `min_df` and `max_df`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000
    ))
])

tfidf_matrix = tfidf_pipeline.fit_transform(df["tags"])
tfidf_pipeline

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=50000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english'))])

### TF-IDF Matrix Shape

In [ ]:
print("TF-IDF Shape:", tfidf_matrix.shape)

TF-IDF Shape: (9828, 32135)


### Movie Lookup Helper

A small helper to find a movie's row index from its title. Used by every recommendation function in the notebook.

In [ ]:
def find_movie(title):
    
    matches = df[
        df["Title"].str.lower().str.strip() ==
        title.lower().strip()
    ]
    
    if matches.empty:
        return None
    
    return matches.index[0]

### TF-IDF Baseline Recommendation Function

Similarity between movies is calculated with cosine similarity on the TF-IDF vectors, and the top N most similar movies (excluding the movie itself) are returned.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def recommend_movies_v2(title, n=10):
    
    # Find the selected movie
    movie_index = find_movie(title)
    
    if movie_index is None:
        print(f"Movie '{title}' not found.")
        return None, None
    
    # Get selected movie details
    selected_movie = df.iloc[[movie_index]][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    # Calculate similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()
    
    # Sort by similarity
    similar_indices = similarity_scores.argsort()[::-1]
    
    # Remove selected movie
    similar_indices = similar_indices[
        similar_indices != movie_index
    ]
    
    # Get top N similar movies
    top_indices = similar_indices[:n]
    
    # Create recommendations
    recommendations = df.iloc[top_indices][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    # Add similarity score
    recommendations["Similarity"] = (
        similarity_scores[top_indices].round(4)
    )
    
    recommendations = recommendations.reset_index(drop=True)
    
    return selected_movie.reset_index(drop=True), recommendations

### Testing TF-IDF Baseline

In [ ]:
recommendations_v2 = recommend_movies_v2("Interstellar", 10)

recommendations_v2

(   movie_id         Title Release_Date                              Genre  \
 0       461  Interstellar   2014-11-05  Adventure, Drama, Science Fiction   
 
    Vote_Average  Popularity                                         Poster_Url  
 0           8.4     110.531  https://image.tmdb.org/t/p/original/gEU2QniE6E...  ,
    movie_id                                  Title Release_Date  \
 0       847               Star Wars: The Last Jedi   2017-12-13   
 1      1399                     Dumb and Dumber To   2014-11-12   
 2      3054                                Gattaca   1997-09-07   
 3      2033                                   Dune   1984-12-14   
 4      6728                       Midnight Special   2016-02-12   
 5      3719                               Stargate   1994-10-27   
 6      1676           Space Pirate Captain Harlock   2013-09-07   
 7      3816                                   Okja   2017-06-28   
 8      3815  Ben 10 Versus the Universe: The Movie   2020-10-02 

# 8. TF-IDF V2

### TF-IDF Vectorization on Weighted Tags

Same TF-IDF setup as the baseline, but built on `tags_v2` (weighted genre) instead of `tags`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_pipeline_v2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000
    ))
])

tfidf_matrix_v2 = tfidf_pipeline_v2.fit_transform(
    df["tags_v2"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix_v2.shape)


TF-IDF Matrix Shape: (9828, 33087)


### TF-IDF V2 Recommendation Function

Same logic as the baseline recommender, using the V2 TF-IDF matrix.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies_v3(title, n=10):
    
    movie_index = find_movie(title)
    
    if movie_index is None:
        print(f"Movie '{title}' not found.")
        return None, None
    
    selected_movie = df.iloc[[movie_index]][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    similarity_scores_v2 = cosine_similarity(
        tfidf_matrix_v2[movie_index],
        tfidf_matrix_v2
    ).flatten()
    
    similar_indices_v2 = similarity_scores_v2.argsort()[::-1]
    
    similar_indices_v2 = similar_indices_v2[
        similar_indices_v2 != movie_index
    ]
    
    top_indices_v2 = similar_indices_v2[:n]
    
    recommendations_v2 = df.iloc[top_indices_v2][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    recommendations_v2["Similarity"] = (
        similarity_scores_v2[top_indices_v2].round(4)
    )
    
    return (
        selected_movie.reset_index(drop=True),
        recommendations_v2.reset_index(drop=True)
    )

### Testing TF-IDF V2

With genre weighted more heavily, recommendations lean more towards movies sharing the same genres, compared to the baseline.

In [ ]:
test_movies = [
    "Interstellar",
    "The Dark Knight",
    "Titanic",
    "The Avengers",
    "Toy Story"
]

for movie_name in test_movies:
    
    print("\n" + "=" * 70)
    print(f"Recommendations for: {movie_name}")
    print("=" * 70)
    
    _, test_results = recommend_movies_v3(movie_name, 5)
    
    print(
        test_results[
            ["Title", "Genre", "Vote_Average", "Similarity"]
        ]
    )


Recommendations for: Interstellar
                          Title                              Genre  \
0              Midnight Special  Adventure, Drama, Science Fiction   
1                          Okja  Adventure, Drama, Science Fiction   
2                The Last Mimzy  Drama, Science Fiction, Adventure   
3  A.I. Artificial Intelligence  Drama, Science Fiction, Adventure   
4                      Last Man             Drama, Science Fiction   

   Vote_Average  Similarity  
0           6.2      0.5711  
1           7.4      0.5340  
2           6.3      0.4588  
3           7.0      0.4284  
4           0.0      0.3751  

Recommendations for: The Dark Knight
                          Title                           Genre  Vote_Average  \
0                     Initial D  Action, Crime, Thriller, Drama           6.7   
1        Assault on Precinct 13  Action, Crime, Thriller, Drama           6.1   
2  The Girl in the Spider's Web  Action, Crime, Thriller, Drama           6.1   
3 

# 9. Hybrid Ranking

TF-IDF similarity alone can surface obscure, low-quality movies just because the text happens to match. The hybrid approach combines:

- TF-IDF V2 similarity (75%)
- normalized rating, `Vote_Average / 10` (15%)
- normalized popularity, min-max scaled (10%)

into a single `Final_Score`, so that well-rated and reasonably popular movies are ranked a bit higher among similar candidates.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def recommend_movies_hybrid(title, n=10):
    
    # --------------------------------------------------
    # Step 1: Validate input
    # --------------------------------------------------
    
    if n <= 0:
        raise ValueError("n must be greater than 0")
    
    
    # --------------------------------------------------
    # Step 2: Find selected movie
    # --------------------------------------------------
    
    movie_index = find_movie(title)
    
    if movie_index is None:
        print(f"Movie '{title}' not found.")
        return None, None
    
    
    # --------------------------------------------------
    # Step 3: Store selected movie
    # --------------------------------------------------
    
    selected_movie = df.iloc[[movie_index]][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    
    # --------------------------------------------------
    # Step 4: Calculate TF-IDF similarity
    # --------------------------------------------------
    
    similarity_scores = cosine_similarity(
        tfidf_matrix_v2[movie_index],
        tfidf_matrix_v2
    ).flatten()
    
    
    # --------------------------------------------------
    # Step 5: Create candidate DataFrame
    # --------------------------------------------------
    
    candidate_indices = np.arange(len(df))
    
    candidate_data = df.iloc[candidate_indices][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    candidate_data["Similarity"] = similarity_scores
    
    
    # --------------------------------------------------
    # Step 6: Remove selected movie
    # --------------------------------------------------
    
    candidate_data = candidate_data[
        candidate_data.index != movie_index
    ]
    
    
    # --------------------------------------------------
    # Step 7: Remove duplicate titles
    # --------------------------------------------------
    
    normalized_title = title.lower().strip()
    
    candidate_data = candidate_data[
        candidate_data["Title"]
        .fillna("")
        .str.lower()
        .str.strip()
        != normalized_title
    ]
    
    
    # --------------------------------------------------
    # Step 8: Clean rating and popularity
    # --------------------------------------------------
    
    candidate_data["Vote_Average"] = (
        pd.to_numeric(
            candidate_data["Vote_Average"],
            errors="coerce"
        )
        .fillna(0)
    )
    
    candidate_data["Popularity"] = (
        pd.to_numeric(
            candidate_data["Popularity"],
            errors="coerce"
        )
        .fillna(0)
    )
    
    
    # --------------------------------------------------
    # Step 9: Normalize rating
    # Rating range is approximately 0-10
    # --------------------------------------------------
    
    candidate_data["Rating_Score"] = (
        candidate_data["Vote_Average"] / 10
    )
    
    
    # --------------------------------------------------
    # Step 10: Normalize popularity
    # Min-Max normalization
    # --------------------------------------------------
    
    popularity_min = candidate_data["Popularity"].min()
    popularity_max = candidate_data["Popularity"].max()
    
    if popularity_max > popularity_min:
        
        candidate_data["Popularity_Score"] = (
            candidate_data["Popularity"] - popularity_min
        ) / (
            popularity_max - popularity_min
        )
    
    else:
        
        candidate_data["Popularity_Score"] = 0
    
    
    # --------------------------------------------------
    # Step 11: Calculate Hybrid Final Score
    # --------------------------------------------------
    
    candidate_data["Final_Score"] = (
        0.75 * candidate_data["Similarity"]
        + 0.15 * candidate_data["Rating_Score"]
        + 0.10 * candidate_data["Popularity_Score"]
    )
    
    
    # --------------------------------------------------
    # Step 12: Sort by Final Score
    # --------------------------------------------------
    
    recommendations = candidate_data.sort_values(
        by="Final_Score",
        ascending=False
    ).head(n)
    
    
    # --------------------------------------------------
    # Step 13: Select final output columns
    # --------------------------------------------------
    
    recommendations = recommendations[
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url",
            "Similarity",
            "Final_Score"
        ]
    ].copy()
    
    
    # --------------------------------------------------
    # Step 14: Round scores
    # --------------------------------------------------
    
    recommendations["Similarity"] = (
        recommendations["Similarity"].round(4)
    )
    
    recommendations["Final_Score"] = (
        recommendations["Final_Score"].round(4)
    )
    
    
    # --------------------------------------------------
    # Step 15: Reset index
    # --------------------------------------------------
    
    recommendations = recommendations.reset_index(drop=True)
    
    
    return (
        selected_movie.reset_index(drop=True),
        recommendations
    )

### Testing Hybrid Ranking

In [ ]:
test_movies_hybrid = [
    "Interstellar",
    "The Dark Knight",
    "Titanic",
    "The Avengers",
    "Toy Story"
]

for movie_name in test_movies_hybrid:
    
    print("\n" + "=" * 70)
    print(f"Recommendations for: {movie_name}")
    print("=" * 70)
    
    _, hybrid_results = recommend_movies_hybrid(
        movie_name,
        5
    )
    
    print(
        hybrid_results[
            [
                "Title",
                "Genre",
                "Vote_Average",
                "Similarity",
                "Final_Score"
            ]
        ].to_string(index=False)
    )


Recommendations for: Interstellar
                       Title                                     Genre  Vote_Average  Similarity  Final_Score
            Midnight Special         Adventure, Drama, Science Fiction           6.2      0.5711       0.5217
                        Okja         Adventure, Drama, Science Fiction           7.4      0.5340       0.5120
              The Last Mimzy         Drama, Science Fiction, Adventure           6.3      0.4588       0.4389
A.I. Artificial Intelligence         Drama, Science Fiction, Adventure           7.0      0.4284       0.4267
          Planet of the Apes Science Fiction, Adventure, Drama, Action           7.6      0.3569       0.3822

Recommendations for: The Dark Knight
                               Title                             Genre  Vote_Average  Similarity  Final_Score
               The Dark Knight Rises    Action, Crime, Drama, Thriller           7.8      0.3426       0.3775
                           Initial D    Action,

# 10. Sentence Transformer

TF-IDF mainly matches movies based on shared words — it has no idea that "space travel" and "interstellar voyage" mean the same thing. A Sentence Transformer instead converts each movie's text into a dense vector that captures its meaning, so movies can be found similar even if they use different words.

### Loading the Sentence Transformer

`all-MiniLM-L6-v2` is a small, fast sentence embedding model, good enough for this use case without needing a GPU.

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


### Generating Movie Embeddings

Each movie's `tags_v2` text is converted into a 384-dimensional semantic vector. Embeddings are normalized so cosine similarity can be computed with a simple dot product later.

In [ ]:
movie_texts = df["tags_v2"].fillna("").astype(str).tolist()

movie_embeddings = embedding_model.encode(
    movie_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding Shape:", movie_embeddings.shape)

Batches:   0%|          | 0/308 [00:00<?, ?it/s]

Embedding Shape: (9828, 384)


### Embedding Shape

In [ ]:
movie_embeddings.shape

(9828, 384)

# 11. FAISS Semantic Search

FAISS is used to search the embedding space efficiently instead of comparing a query against every movie one by one with plain cosine similarity. Since the embeddings are already normalized, `IndexFlatIP` (inner product) gives the same ranking as cosine similarity, but with fast, optimized search.

In [ ]:
import faiss
import numpy as np
embedding_dimension = movie_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(
    np.asarray(movie_embeddings, dtype="float32")
)

print("FAISS index created.")
print("Number of movies:", faiss_index.ntotal)

FAISS index created.
Number of movies: 9828


### FAISS Index Size

In [ ]:
faiss_index.ntotal

9828

# 12. Semantic Recommendation Testing

### Semantic Recommendation Function

For a given movie, its embedding is used as the query vector and FAISS returns the closest movies in the embedding space.

In [ ]:
def semantic_recommend_movies(title, n=10):
    
    movie_index = find_movie(title)
    
    if movie_index is None:
        print(f"Movie '{title}' not found.")
        return None, None
    
    selected_movie = df.iloc[[movie_index]][
        [
            "movie_id",
            "Title",
            "Release_Date",
            "Genre",
            "Vote_Average",
            "Popularity",
            "Poster_Url"
        ]
    ].copy()
    
    query_vector = np.asarray(
        movie_embeddings[movie_index],
        dtype="float32"
    ).reshape(1, -1)
    
    search_scores, search_indices = faiss_index.search(
        query_vector,
        n + 10
    )
    
    results = []
    
    for score, idx in zip(
        search_scores[0],
        search_indices[0]
    ):
        
        if idx == movie_index:
            continue
        
        if (
            df.iloc[idx]["Title"]
            .lower()
            .strip()
            == title.lower().strip()
        ):
            continue
        
        results.append({
            "movie_id": df.iloc[idx]["movie_id"],
            "Title": df.iloc[idx]["Title"],
            "Release_Date": df.iloc[idx]["Release_Date"],
            "Genre": df.iloc[idx]["Genre"],
            "Vote_Average": df.iloc[idx]["Vote_Average"],
            "Popularity": df.iloc[idx]["Popularity"],
            "Poster_Url": df.iloc[idx]["Poster_Url"],
            "Semantic_Similarity": round(float(score), 4)
        })
        
        if len(results) == n:
            break
    
    recommendations = pd.DataFrame(results)
    
    return (
        selected_movie.reset_index(drop=True),
        recommendations.reset_index(drop=True)
    )

### Testing Semantic Recommendations

In [ ]:
test_movies_semantic = [
    "Interstellar",
    "The Dark Knight",
    "Titanic",
    "The Avengers",
    "Toy Story"
]

for test_title in test_movies_semantic:

    _, semantic_output = semantic_recommend_movies(
        test_title,
        5
    )

    print(f"\n{'=' * 60}")
    print(f"Recommendations for: {test_title}")
    print(f"{'=' * 60}")

    display(
        semantic_output[
            [
                "Title",
                "Genre",
                "Vote_Average",
                "Semantic_Similarity"
            ]
        ]
    )


Recommendations for: Interstellar


,Title,Genre,Vote_Average,Semantic_Similarity
0,Prometheus,"Science Fiction, Adventure, Mystery",6.5,0.7612
1,Passengers,"Drama, Romance, Science Fiction",6.9,0.7111
2,Star Trek Beyond,"Action, Adventure, Science Fiction",6.8,0.7051
3,Moontrap: Target Earth,"Action, Science Fiction, Adventure",4.6,0.6954
4,Lost in Space,"Science Fiction, Adventure",5.3,0.6952



Recommendations for: The Dark Knight


,Title,Genre,Vote_Average,Semantic_Similarity
0,The Dark Knight Rises,"Action, Crime, Drama, Thriller",7.8,0.8546
1,Batman Begins,"Action, Crime, Drama",7.7,0.7805
2,"Batman: The Long Halloween, Part One","Animation, Mystery, Action, Crime",7.8,0.7801
3,"Batman: The Long Halloween, Part Two","Animation, Mystery, Action, Crime",7.7,0.7705
4,Joker,"Crime, Thriller, Drama",8.2,0.7700



Recommendations for: Titanic


,Title,Genre,Vote_Average,Semantic_Similarity
0,Message in a Bottle,"Romance, Drama",6.1,0.6085
1,Summer of 85,"Romance, Drama",7.4,0.5978
2,A Night to Remember,"Drama, Action, History",7.5,0.5969
3,Kids in Love,"Drama, Romance",6.1,0.5909
4,Orbiter 9,"Drama, Science Fiction, Romance",5.8,0.5906



Recommendations for: The Avengers


,Title,Genre,Vote_Average,Semantic_Similarity
0,Avengers: Age of Ultron,"Action, Adventure, Science Fiction",7.3,0.7456
1,Captain America: The Winter Soldier,"Action, Adventure, Science Fiction",7.7,0.7039
2,Universal Soldier: Regeneration,"Adventure, Action, Science Fiction, Thriller",5.6,0.7012
3,Iron Man 3,"Action, Adventure, Science Fiction",6.9,0.6994
4,Avengers: Infinity War,"Adventure, Action, Science Fiction",8.3,0.6964



Recommendations for: Toy Story


,Title,Genre,Vote_Average,Semantic_Similarity
0,Toy Story 2,"Animation, Comedy, Family",7.6,0.8266
1,Toy Story 3,"Animation, Family, Comedy",7.8,0.7783
2,Toy Story 4,"Family, Adventure, Animation, Comedy, Fantasy,...",7.6,0.7737
3,Buzz Lightyear of Star Command: The Adventure ...,"Animation, Family, Comedy, Science Fiction, Ad...",6.4,0.6972
4,Hawaiian Vacation,"Animation, Family, Comedy",7.0,0.6679


# 13. Model Comparison

| Approach | What it captures | Observation from the tests above |
|---|---|---|
| TF-IDF V1 | exact word overlap in overview + genre + language | picks up loosely related movies, some recommendations feel like word-matching coincidences |
| TF-IDF V2 | same as V1, with genre weighted more | recommendations lean more strongly towards the same genre |
| Hybrid | TF-IDF V2 similarity blended with rating and popularity | similar movies to V2, but nudges well-rated, more popular titles higher |
| Sentence Transformer + FAISS | semantic meaning of the movie description | recommendations for Interstellar, The Dark Knight, Titanic, The Avengers and Toy Story are noticeably more thematically consistent, not just word-matched |

No formal evaluation metrics (precision/recall, etc.) were computed since there are no ground-truth labels for "correct" recommendations in this dataset — the comparison above is based on manually reviewing the test results.

**Sentence Transformer + FAISS is used as the primary recommendation approach** going forward, since it produced better semantic recommendations than the TF-IDF-based approaches in every tested example. TF-IDF and the hybrid ranking are kept in the notebook as baseline/experimental approaches that show how the project evolved.

# 14. Save Model Artifacts

### Saving Movie Embeddings

The generated Sentence Transformer embeddings are saved so they don't need to be regenerated every time the application starts.

In [ ]:
np.save("../models/movie_embeddings.npy", movie_embeddings)
print("Movie embeddings saved successfully.")

### Saving the FAISS Index

The FAISS index is saved separately so it can be reloaded directly instead of rebuilding it from the embeddings each time.

In [ ]:
faiss.write_index(faiss_index, "../models/faiss_index.bin")
print("FAISS index saved successfully.")

### Saving Movie Metadata

The metadata columns needed to display a recommended movie (title, overview, genre, rating, poster, etc.) are saved to CSV, keyed by `movie_id`.

In [ ]:
metadata_columns = [
    "movie_id",
    "Title",
    "Release_Date",
    "Overview",
    "Popularity",
    "Vote_Count",
    "Vote_Average",
    "Original_Language",
    "Genre",
    "Poster_Url"
]

movie_metadata = df[metadata_columns].copy()

movie_metadata.to_csv(
    "../models/movie_metadata.csv",
    index=False
)

print("Movie metadata saved successfully.")
print("Shape:", movie_metadata.shape)

Movie metadata saved successfully.
Shape: (9828, 10)


# 15. Load and Verify Saved Artifacts

### Loading Saved Embeddings

In [ ]:
import numpy as np
import os

embedding_file_path = "../models/movie_embeddings.npy"

loaded_movie_embeddings = np.load(
    embedding_file_path
)

print("Embeddings loaded successfully.")
print("Shape:", loaded_movie_embeddings.shape)

Embeddings loaded successfully.
Shape: (9828, 384)


### Loading Saved FAISS Index

In [ ]:
import faiss

faiss_file_path = "../models/faiss_index.bin"

loaded_faiss_index = faiss.read_index(
    faiss_file_path
)

print("FAISS index loaded successfully.")
print("Number of vectors:", loaded_faiss_index.ntotal)

FAISS index loaded successfully.
Number of vectors: 9828


### Verifying Saved Artifacts

Check that the number of stored embeddings matches the number of vectors in the FAISS index.

In [ ]:
print("Embedding count:", len(loaded_movie_embeddings))
print("FAISS vector count:", loaded_faiss_index.ntotal)

Embedding count: 9828
FAISS vector count: 9828


# 16. Movie Metadata

`movie_metadata` (and the saved `movie_metadata.csv`) is what the application uses to display full movie information — title, overview, genre, rating, and poster — once a recommendation has been made. The project does not call an external movie API; the `Poster_Url` column already in the dataset is used directly.

In [ ]:
movie_metadata[
    movie_metadata["Title"]
    .str.lower()
    .str.strip()
    == "prometheus"
]

,movie_id,Title,Release_Date,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
618,618,Prometheus,2012-05-30,A team of explorers discover a clue to the ori...,91.808,10081.0,6.5,en,"Science Fiction, Adventure, Mystery",https://image.tmdb.org/t/p/original/m7nZCtHJyD...


# 17. Movie Details Function

`get_movie_details(movie_id)` looks up the full metadata for one movie by its id.

In [ ]:
def get_movie_details(movie_id):
    movie_row = movie_metadata[
        movie_metadata["movie_id"] == movie_id
    ]

    if movie_row.empty:
        return None

    return movie_row.iloc[0].to_dict()

### Testing Movie Details Function

In [ ]:
prometheus_details = get_movie_details(618)

prometheus_details

{'movie_id': 618,
 'Title': 'Prometheus',
 'Release_Date': '2012-05-30',
 'Overview': 'A team of explorers discover a clue to the origins of mankind on Earth, leading them on a journey to the darkest corners of the universe. There, they must fight a terrifying battle to save the future of the human race.',
 'Popularity': 91.808,
 'Vote_Count': 10081.0,
 'Vote_Average': 6.5,
 'Original_Language': 'en',
 'Genre': 'Science Fiction, Adventure, Mystery',
 'Poster_Url': 'https://image.tmdb.org/t/p/original/m7nZCtHJyDLncBUarfM5h5mrppx.jpg'}

# 18. Final Recommendation Function

`get_recommended_movie_details(title, n=10)` is the final, reusable recommendation function. It:

1. finds the requested movie
2. gets its embedding
3. searches the FAISS index
4. removes the selected movie from the results
5. removes duplicate same-title entries
6. retrieves full movie metadata for each result
7. returns complete recommendation information
8. includes the semantic similarity score for each recommendation

In [ ]:
def get_recommended_movie_details(title, n=10):

    movie_index = find_movie(title)

    if movie_index is None:
        print(f"Movie '{title}' not found.")
        return None, []

    query_vector = np.asarray(
        loaded_movie_embeddings[movie_index],
        dtype="float32"
    ).reshape(1, -1)

    search_scores, search_indices = loaded_faiss_index.search(
        query_vector,
        n + 10
    )

    recommended_movies = []

    normalized_title = title.lower().strip()

    for score, movie_idx in zip(
        search_scores[0],
        search_indices[0]
    ):

        current_title = (
            str(movie_metadata.iloc[movie_idx]["Title"])
            .lower()
            .strip()
        )

        if movie_idx == movie_index:
            continue

        if current_title == normalized_title:
            continue

        movie_details = movie_metadata.iloc[movie_idx].to_dict()

        movie_details["Semantic_Similarity"] = round(
            float(score),
            4
        )

        recommended_movies.append(movie_details)

        if len(recommended_movies) == n:
            break

    selected_movie_details = (
        movie_metadata.iloc[movie_index]
        .to_dict()
    )

    return selected_movie_details, recommended_movies

# 19. Final Test

One end-to-end test of the complete pipeline, using the final function.

In [ ]:
selected_movie, recommended_movie_list = (
    get_recommended_movie_details(
        "Interstellar",
        5
    )
)

selected_movie

{'movie_id': 461,
 'Title': 'Interstellar',
 'Release_Date': '2014-11-05',
 'Overview': 'The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.',
 'Popularity': 110.531,
 'Vote_Count': 27975.0,
 'Vote_Average': 8.4,
 'Original_Language': 'en',
 'Genre': 'Adventure, Drama, Science Fiction',
 'Poster_Url': 'https://image.tmdb.org/t/p/original/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg'}

### Recommended Movies

In [ ]:
recommended_movie_list

[{'movie_id': 618,
  'Title': 'Prometheus',
  'Release_Date': '2012-05-30',
  'Overview': 'A team of explorers discover a clue to the origins of mankind on Earth, leading them on a journey to the darkest corners of the universe. There, they must fight a terrifying battle to save the future of the human race.',
  'Popularity': 91.808,
  'Vote_Count': 10081.0,
  'Vote_Average': 6.5,
  'Original_Language': 'en',
  'Genre': 'Science Fiction, Adventure, Mystery',
  'Poster_Url': 'https://image.tmdb.org/t/p/original/m7nZCtHJyDLncBUarfM5h5mrppx.jpg',
  'Semantic_Similarity': 0.7612},
 {'movie_id': 1765,
  'Title': 'Passengers',
  'Release_Date': '2016-12-21',
  'Overview': 'A spacecraft traveling to a distant colony planet and transporting thousands of people has a malfunction in its sleep chambers. As a result, two passengers are awakened 90 years early.',
  'Popularity': 44.528,
  'Vote_Count': 11236.0,
  'Vote_Average': 6.9,
  'Original_Language': 'en',
  'Genre': 'Drama, Romance, Science 

# 20. Final Project Summary

CineMind is a content-based semantic movie recommendation system. The project began with TF-IDF-based similarity as a classical baseline, then improved it by weighting genre more heavily (TF-IDF V2) and by blending similarity with rating and popularity (hybrid ranking).

The representation was then upgraded using a Sentence Transformer to capture the semantic meaning of each movie's description, instead of just matching words. The resulting embeddings were normalized and indexed with FAISS for fast similarity search.

The final system takes a movie title, finds semantically similar movies through FAISS, and combines them with locally stored movie metadata (title, genre, rating, overview and poster URL) to return complete recommendation results.